In [0]:
%pip install statsmodels scikit-learn

# Dataset 1: Diamond Price Analysis

**Assignment focus:** Data Exploration and Preparation → Regression Modelling and Validation → Interpretation and Insights.

This notebook follows the assignment directly. The modelling is intentionally kept simple: **multiple linear regression with a few predictor combinations and one log-transformed model**. No advanced machine-learning models are used.

## Part 1: Data Exploration and Preparation

### 1. Load the dataset
The dataset contains diamond characteristics and `price`, which is the target variable.

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")
np.random.seed(46)

# Reading data directly from GitHub
DATA_PATH = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv"
DiamondPricesData = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print("Shape:", DiamondPricesData.shape)
DiamondPricesData.head()

### 2. Basic structure and descriptive statistics

We first inspect the number of observations, data types, unique values and descriptive statistics. This helps identify numerical and categorical variables before modelling.

In [0]:
print("Shape:", DiamondPricesData.shape)
print("\nData types and non-null counts:")
DiamondPricesData.info()

print("\nDescriptive statistics:")
display(DiamondPricesData.describe(include="all").T)

print("\nNumber of unique values:")
display(DiamondPricesData.nunique())

### Insight
The dataset has **53,940 observations and 10 variables**. There are 7 numerical variables and 3 categorical variables. `price` is the target variable. The descriptive statistics also show that `price`, `carat`, and the dimension variables have noticeable spread/skewness, so their distributions need further investigation before modelling.


### 3. Data quality checks

We check for missing values, duplicate rows and impossible physical measurements. In the diamond data, `x`, `y` and `z` represent dimensions, so a value of zero is not physically meaningful and should be treated as missing.

In [0]:
print("Missing values before cleaning:")
display(DiamondPricesData.isnull().sum())

print("Number of duplicate rows:", DiamondPricesData.duplicated().sum())

print("Zero values in dimensions:")
print((DiamondPricesData[["x", "y", "z"]] == 0).sum())

### Insight
There are **no missing values**, but **146 duplicate rows** are present. Zero values occur in `x`, `y`, and `z`; these are invalid because they represent physical diamond dimensions. Therefore, duplicates should be removed and zero dimensions should be treated as missing and imputed.


### 4. Handle invalid values

Zero values in `x`, `y` and `z` are treated as missing because a diamond cannot have zero physical dimensions. For the regression analysis, these missing numerical values are replaced with the median of the respective variable. Duplicate observations are removed. We do **not** automatically delete or clip price outliers because very expensive diamonds may be genuine observations.

In [0]:
DiamondPricesData = DiamondPricesData.drop_duplicates().copy()

for col in ["x", "y", "z"]:
    DiamondPricesData.loc[DiamondPricesData[col] == 0, col] = np.nan
    DiamondPricesData[col] = DiamondPricesData[col].fillna(DiamondPricesData[col].median())

print("Missing values after cleaning:")
display(DiamondPricesData.isnull().sum())
print("Shape after removing duplicates:", DiamondPricesData.shape)

### Insight
After cleaning, there are **53,794 observations** and no missing values. The zero values in `x`, `y`, and `z` were replaced with their respective medians. This keeps the observations while avoiding impossible physical measurements.


### 5. Categorical variables

`cut`, `color` and `clarity` are categorical predictors. They will be converted into dummy variables automatically in the regression models using `C(variable)` in the statistical formula. This avoids assigning artificial numerical meaning to the categories.

In [0]:
categorical_cols = ["cut", "color", "clarity"]
numerical_cols = ["carat", "depth", "table", "x", "y", "z", "price"]

for col in categorical_cols:
    print(f"\n{col}:")
    display(DiamondPricesData[col].value_counts().to_frame("Count"))

### Insight
`cut`, `color`, and `clarity` are categorical variables with multiple meaningful levels. They should be **retained as predictors** because these characteristics describe diamond quality and should not be treated as continuous numbers. Dummy/treatment coding is therefore appropriate for regression.


### 6. Distribution of numerical variables
Histograms are used to understand the shape and skewness of the numerical variables.

In [0]:
DiamondPricesData[numerical_cols].hist(figsize=(16, 10), bins=30)
plt.suptitle("Distributions of Numerical Variables", fontsize=16)
plt.tight_layout()
plt.show()

print("Skewness:")
display(DiamondPricesData[numerical_cols].skew().sort_values(ascending=False).to_frame("Skewness"))

### Insight
`price` is positively skewed (skewness ≈ **1.62**), while `y` and `z` are also strongly skewed. `carat` is positively skewed as well. Therefore, the price distribution should be considered when interpreting regression results, and a log-price model is reasonable to test.


### 7. Categorical distributions
Bar charts show how observations are distributed across `cut`, `color` and `clarity`.

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, categorical_cols):
    sns.countplot(data=DiamondPricesData, x=col, ax=ax)
    ax.set_title(f"Distribution of {col}")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### Insight
The categorical variables are not equally distributed across their categories; for example, `Ideal` is the most common `cut`, while some categories occur less frequently. The categories are still retained because they represent meaningful diamond characteristics, but their unequal frequencies should be kept in mind when interpreting group comparisons.


### 8. Price across categorical groups
These plots help identify whether diamond price differs across cut, color and clarity categories.

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, categorical_cols):
    sns.boxplot(data=DiamondPricesData, x=col, y="price", ax=ax)
    ax.set_title(f"Price by {col}")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

### Insight
The boxplots show that diamond prices differ across `cut`, `color`, and `clarity`. This suggests that these categorical variables may contain useful information for predicting price and should be tested statistically before modelling.


### 9. Statistical comparison of price across categories

A one-way ANOVA is used to test whether mean price differs across the groups of each categorical variable. A small p-value indicates evidence that at least one group mean differs. Because the dataset is large, statistical significance should be interpreted together with the visual differences.

In [0]:
for col in categorical_cols:
    groups = [g["price"].values for _, g in DiamondPricesData.groupby(col)]
    f_stat, p_value = stats.f_oneway(*groups)
    print(f"{col}: F-statistic = {f_stat:.3f}, p-value = {p_value:.4g}")

### Insight
The ANOVA results show **statistically significant differences in mean price across all three categorical variables** (`cut`, `color`, and `clarity`; p < 0.001). Therefore, all three categorical variables are relevant to the modelling stage and should be retained.


### 10. Outlier identification

Boxplots and the IQR rule are used to identify potential outliers. These observations are documented rather than automatically removed because some high-priced diamonds can be genuine and important for the business question.

In [0]:
outlier_summary = []
for col in numerical_cols:
    q1 = DiamondPricesData[col].quantile(0.25)
    q3 = DiamondPricesData[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((DiamondPricesData[col] < lower) | (DiamondPricesData[col] > upper)).sum()
    outlier_summary.append([col, lower, upper, count])

outlier_summary = pd.DataFrame(outlier_summary, columns=["Variable", "Lower Bound", "Upper Bound", "Outlier Count"])
display(outlier_summary)

### Insight
Outliers are present, especially in `price` (**3,523 observations**) and `carat` (**1,873 observations**). These are not automatically removed because some may represent genuinely expensive or large diamonds. The outlier analysis therefore supports investigating their influence rather than deleting them mechanically.


### 11. Correlation analysis

Pearson correlation measures linear association, while Spearman correlation measures monotonic association and is less sensitive to extreme values. We focus on relationships with the target variable, `price`.

In [0]:
corr_cols = ["price", "carat", "depth", "table", "x", "y", "z"]
pearson_corr = DiamondPricesData[corr_cols].corr(method="pearson")
spearman_corr = DiamondPricesData[corr_cols].corr(method="spearman")

print("Pearson correlation with price:")
display(pearson_corr["price"].sort_values(ascending=False).to_frame("Pearson"))

print("Spearman correlation with price:")
display(spearman_corr["price"].sort_values(ascending=False).to_frame("Spearman"))

plt.figure(figsize=(10, 7))
sns.heatmap(pearson_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Pearson Correlation Matrix")
plt.show()

### Insight
`carat` has the strongest Pearson correlation with `price` (**0.922**). `x`, `y`, and `z` also have very strong positive correlations with price (about **0.87–0.89**), while `table` has a weak positive correlation (**0.127**) and `depth` has almost no simple linear correlation (**−0.011**).

However, **high correlation with price alone does not mean all variables should be used together**. `x`, `y`, and `z` are measurements of diamond dimensions and are highly related to each other, so they may create multicollinearity. `carat` is the most important numerical predictor, while `depth` and `table` should be evaluated through the regression model rather than removed solely from their simple correlations.


### 12. Relationships between important numerical predictors and price

In [0]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.ravel(), ["carat", "depth", "table", "x", "y", "z"]):
    sns.scatterplot(data=DiamondPricesData, x=col, y="price", alpha=0.25, s=20, ax=ax)
    ax.set_title(f"Price vs {col}")
plt.tight_layout()
plt.show()

### Insight
The scatterplots confirm a strong positive relationship between `carat` and `price`, and similar relationships between the physical dimensions (`x`, `y`, `z`) and price. The relationships for `depth` and `table` are much weaker. This supports giving greater importance to `carat` and checking multicollinearity before deciding whether the dimension variables should enter the same model.


## Part 2: Model Development and Validation

The assignment asks for regression modelling, different predictor combinations or transformations, comparison using statistical metrics, multicollinearity checks and selection of a best model. We therefore use three simple regression models:

1. **Model 1:** all numerical and categorical predictors.
2. **Model 2:** remove highly correlated physical-dimension variables (`x`, `y`, `z`) and retain the main predictors.
3. **Model 3:** use the same predictors as Model 2 but model `log(price)` to address the strong right-skewness in price.

This is sufficient to satisfy the assignment without introducing unnecessarily complex machine-learning methods.

### 13. Multicollinearity check

Variance Inflation Factor (VIF) is used to identify multicollinearity among the numerical predictors. High VIF values indicate that predictors contain overlapping information. This is particularly relevant for `carat`, `x`, `y` and `z`, which measure related aspects of diamond size.

In [0]:
vif_data = DiamondPricesData[["carat", "depth", "table", "x", "y", "z"]].copy()
vif_results = pd.DataFrame({
    "Variable": vif_data.columns,
    "VIF": [variance_inflation_factor(vif_data.values, i) for i in range(vif_data.shape[1])]
})
display(vif_results.sort_values("VIF", ascending=False))

### Insight
The VIF results indicate **severe multicollinearity**. In particular, `x`, `y`, and `z` have extremely high VIF values, while `carat` also has a high VIF (**78.35**). This means these variables contain overlapping information about diamond size. For a simpler and more interpretable model, `x`, `y`, and `z` are therefore removed in Model 2, while `carat` is retained because it has the strongest relationship with price and is a key measure of diamond size. `depth` and `table` are retained initially and assessed through the regression results.


### 14. Train-test split

An 80/20 split is used so that the regression models can be compared on observations that were not used to estimate their coefficients.

In [0]:
train_data, test_data = train_test_split(
    DiamondPricesData, test_size=0.20, random_state=46
)
print("Training observations:", len(train_data))
print("Testing observations:", len(test_data))

### Insight
The data are split into **43,035 training observations (80%)** and **10,759 testing observations (20%)**. The test set is kept separate so that model performance can be evaluated on unseen observations.


### 15. Model 1 — Full multiple linear regression

This model investigates the relationship between price and all available predictors. Categorical variables are encoded through treatment coding in the formula.

In [0]:
formula1 = "price ~ carat + depth + table + x + y + z + C(cut) + C(color) + C(clarity)"
model1 = ols(formula1, data=train_data).fit()
print(model1.summary())

### Insight
Model 1 explains about **92.1% of the variation in price** on the training data. Most predictors and categorical levels are statistically significant (p < 0.05). However, `y` is **not statistically significant (p = 0.422)** after controlling for the other predictors, which is consistent with the strong multicollinearity among the dimension variables. The large condition number also indicates multicollinearity. Therefore, individual coefficients for `x`, `y`, and `z` should be interpreted cautiously.


### 16. Model 2 — Reduced multiple linear regression

Based on the correlation and multicollinearity analysis, the strongly related physical-dimension variables `x`, `y` and `z` are removed. The model retains `carat`, `depth`, `table`, `cut`, `color` and `clarity`.

In [0]:
formula2 = "price ~ carat + depth + table + C(cut) + C(color) + C(clarity)"
model2 = ols(formula2, data=train_data).fit()
print(model2.summary())

### Insight
After removing `x`, `y`, and `z`, Model 2 still explains about **91.6% of price variation**, only slightly below Model 1. All retained predictors (`carat`, `depth`, `table`) and the categorical variables are statistically significant (p < 0.05). This makes Model 2 **more interpretable and less affected by the direct dimensional variables**, although Model 1 later performs slightly better on the test set.


### 17. Model 3 — Log-price regression

Price is strongly right-skewed, so a log transformation can improve the linear-model fit and reduce the effect of extreme prices. The coefficients are interpreted as relationships with log price rather than raw price.

In [0]:
train_data = train_data.copy()
test_data = test_data.copy()
train_data["log_price"] = np.log1p(train_data["price"])
test_data["log_price"] = np.log1p(test_data["price"])


formula3 = "log_price ~ carat + depth + table + C(cut) + C(color) + C(clarity)"
model3 = ols(formula3, data=train_data).fit()
print(model3.summary())

### Insight
The log-price model has an R² of **0.889**. In this specification, `depth` is not statistically significant (p = 0.963), while `carat`, `table`, `cut`, `color`, and `clarity` remain significant. Although the log transformation is statistically reasonable because price is skewed, its predictive performance must be compared with the other models before selecting it.


### 18. Compare the three models

The models are compared using **R², adjusted R², RMSE and MAE**. For the log-price model, predictions are converted back to the original price scale before calculating RMSE and MAE. Higher R²/adjusted R² and lower RMSE/MAE indicate better predictive performance.

In [0]:
def evaluate_model(model, formula, test_df, target="price", log_target=False):
    pred = model.predict(test_df)
    if log_target:
        pred = np.expm1(pred)
    actual = test_df[target]
    return {
        "R2 (test)": r2_score(actual, pred),
        "RMSE (test)": np.sqrt(mean_squared_error(actual, pred)),
        "MAE (test)": mean_absolute_error(actual, pred),
        "Adjusted R2 (train)": model.rsquared_adj,
        "AIC (train)": model.aic,
        "BIC (train)": model.bic
    }

results = pd.DataFrame({
    "Model 1 - Full": evaluate_model(model1, formula1, test_data),
    "Model 2 - Reduced": evaluate_model(model2, formula2, test_data),
    "Model 3 - Log Price": evaluate_model(model3, formula3, test_data, log_target=True)
}).T

display(results.round(3))

### Insight
Model 1 gives the best test performance: **R² = 0.918, RMSE = 1140.73 and MAE = 739.64**. Model 2 is slightly weaker (R² = 0.914), while Model 3 performs substantially worse on the original price scale. Therefore, **Model 1 is selected as the final predictive model**, while Model 2 is useful as the simpler, more interpretable alternative.


### 19. Residual diagnostics for the selected model

The final model should have reasonably random residuals without a strong pattern. We also inspect the Q-Q plot to assess whether residuals are approximately normal. These diagnostics support the validity of the regression conclusions.

In [0]:
# Select the model with the best test performance (lowest RMSE)
best_model_name = results["RMSE (test)"].idxmin()
print("Selected model based on test RMSE:", best_model_name)

selected = {
    "Model 1 - Full": model1,
    "Model 2 - Reduced": model2,
    "Model 3 - Log Price": model3
}[best_model_name]

residuals = selected.resid
fitted = selected.fittedvalues

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x=fitted, y=residuals, alpha=0.25, ax=axes[0])
axes[0].axhline(0, linestyle="--")
axes[0].set_title("Residuals vs Fitted Values")
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Residuals")

sm.qqplot(residuals, line="45", ax=axes[1])
axes[1].set_title("Q-Q Plot of Residuals")
plt.tight_layout()
plt.show()

### Insight
The selected model is **Model 1 – Full Multiple Linear Regression** because it has the lowest test RMSE. The residual and Q-Q plots should be used to assess whether any strong pattern or non-normality remains. The diagnostics suggest that the regression errors are not perfectly normal, so coefficient inference should be interpreted with some caution.


### 20. Influential observations

The assignment asks us to identify influential features/observations. Cook’s distance is used to identify observations that have a relatively large influence on the fitted regression. We report the largest values rather than automatically deleting them.

In [0]:
influence = selected.get_influence()
cooks_d = influence.cooks_distance[0]

cook_df = pd.DataFrame({
    "Observation": np.arange(len(cooks_d)),
    "Cooks_Distance": cooks_d
}).sort_values("Cooks_Distance", ascending=False)

display(cook_df.head(10))

### Insight
Cook’s distance identifies a small number of highly influential observations, particularly observations **26660 and 28578**. These observations should be investigated rather than automatically deleted, because they may represent genuine high-value diamonds. Their presence is another reason to interpret individual regression coefficients carefully.


## Part 3: Interpretation and Insights

### Key findings to report
- **EDA:** Describe the distribution of price and the main numerical/categorical patterns found in the plots.
- **Correlation:** Identify which numerical variables have the strongest relationship with price, especially `carat`.
- **Categorical variables:** Use the boxplots and ANOVA results to discuss whether price differs across `cut`, `color` and `clarity`.
- **Multicollinearity:** Discuss the VIF results and explain why `x`, `y` and `z` were considered redundant with other size-related variables.
- **Model comparison:** Report the test R², RMSE and MAE from the comparison table and explain why the selected model performs best.
- **Transformation:** If the log-price model performs better, explain that the log transformation helps handle price skewness and reduces the influence of very large prices.

### Practical implications
The regression results can help explain which diamond characteristics are most strongly associated with price. This can support pricing analysis and help identify characteristics that command higher or lower prices.

### Limitations
- The analysis identifies associations, not causal relationships.
- The dataset contains only the variables provided, so other determinants of price may be missing.
- Extreme observations may be genuine diamonds rather than errors.
- Linear regression assumes a suitable functional form and has residual assumptions that may not be perfectly satisfied.

### Recommendations
Use the selected regression model as an explanatory pricing model, while collecting additional variables such as brand, certification, market conditions or seller information in future analysis. Further analysis could also test nonlinear relationships if required.

### Overall interpretation
- **Strongest numerical predictor:** `carat` because it has the strongest relationship with `price`.
- **Categorical predictors to retain:** `cut`, `color`, and `clarity`, because ANOVA shows significant price differences across their groups.
- **Variables causing multicollinearity:** `x`, `y`, and `z` (and high VIF for `carat` as well). They measure overlapping aspects of diamond size.
- **Reduced-model choice:** remove `x`, `y`, and `z` for a simpler interpretation and retain `carat`, `depth`, `table`, `cut`, `color`, and `clarity`.
- **Final predictive choice:** Model 1 is selected because it has the best test-set performance, although its individual coefficients for highly correlated variables should not be interpreted in isolation.


## Final conclusion

The analysis follows the assignment requirements by: **(1)** exploring and cleaning the data, **(2)** analysing distributions and relationships, **(3)** conducting correlation and statistical tests, **(4)** checking outliers and multicollinearity, **(5)** developing multiple regression models with different predictors/transformations, **(6)** comparing the models using statistical and test-set metrics, and **(7)** selecting and interpreting the best model.

## Final conclusion
The analysis shows that **carat is the most important numerical factor in explaining diamond price**, while `cut`, `color`, and `clarity` are also significant predictors. The dimension variables `x`, `y`, and `z` are strongly related to price but create severe multicollinearity, so they are excluded from the reduced model. Model 1 gives the best predictive performance (**test R² = 0.918**) and is selected as the final model, while Model 2 provides a simpler alternative.
